# Per-UID LSTM for IEEE-CIS Fraud — PyTorch version

This notebook mirrors `Time_Series_LSTM_per_UID.ipynb` (Keras) but is written in
PyTorch, following the structural pattern from the Kaggle reference:

> [arunmohan003 — *Sentiment analysis using LSTM - PyTorch*](https://www.kaggle.com/code/arunmohan003/sentiment-analysis-using-lstm-pytorch)

Same upstream logic as the Keras notebook:

- Load preprocessed checkpoints (`X_train_copy4.pkl`, `X_test_copy4.pkl`, `y_train.pkl`).
- Bridge train+test before windowing so test rows can use train history.
- Add three time-gap features per UID.
- Standardize (fit on train rows only).
- Build per-UID sliding windows of length `WINDOW`.
- Train under **strict expanding-window time validation** with `MIN_TRAIN_MONTHS = 3`.
- Save OOF and test predictions for ensembling with your XGBoost OOF.

## What's adapted from the sentiment-analysis reference

| arunmohan003's notebook | This notebook |
|--|--|
| Word indices `(batch, seq_len)` → `nn.Embedding` → `(batch, seq_len, embed_dim)` | Numeric features `(batch, seq_len, n_features)` directly into LSTM (no embedding) |
| `vocab_size`, `embedding_dim` hyperparameters | `n_features` only — no vocab |
| `nn.LSTM(input_size=embedding_dim, ...)` | `nn.LSTM(input_size=n_features, ...)` |
| `model.init_hidden(batch_size)` per iteration | Same pattern, kept for fidelity |
| `nn.BCELoss` after sigmoid | Same |
| Manual training loop with `optimizer.zero_grad()`, `loss.backward()`, `clip_grad_norm_`, `optimizer.step()` | Same |
| Best model saved by validation loss | Best model saved by **validation AUC** (better metric for fraud) |

The reason there's no embedding layer: in sentiment analysis each word is a discrete
token that has to be turned into a continuous vector. Your fraud features are already
continuous after `StandardScaler` (and previously-categorical fields like `card1` were
already integer-encoded by the upstream pipeline), so the LSTM can ingest them directly.


## MPS / Apple Silicon GPU notes

PyTorch supports Apple Silicon GPU via the **MPS** (Metal Performance Shaders) backend.
The config cell below picks the best available device automatically: `mps` if you're on
M-series, else `cuda`, else `cpu`.

A few specifics for MPS:

- Some ops fall back to CPU silently. For LSTM this works but you may see warnings
  about unsupported dtypes — mostly harmless.
- `torch.compile(...)` doesn't help much on MPS yet (Metal backend is limited),
  so this notebook doesn't use it.
- Mixed-precision (`autocast`) is supported but not used here — fp32 is more stable
  for LSTM on MPS, and the speed difference is small.
- `num_workers > 0` in `DataLoader` can deadlock on macOS in Jupyter. We use
  `num_workers=0` and rely on the unified-memory architecture for fast host-device
  transfer.


In [ ]:
import sys, os
print(sys.executable)
print(os.environ.get("CONDA_DEFAULT_ENV"))

In [ ]:
# 0. Imports and config — PyTorch + MPS-aware
import os, gc, math, time, datetime, warnings, copy
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import StandardScaler

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader

# ----- Configuration -----

WINDOW              = 20      # was 5  (change F)
MIN_TRAIN_MONTHS    = 3
BATCH               = 1024
EPOCHS              = 30      # was 12 (change I)
LR                  = 1e-3
WEIGHT_DECAY        = 2e-4
GRAD_CLIP           = 1.0     # was 0.5 (change D)
EARLY_STOP_PATIENCE = 6
SEED                = 42

HIDDEN_DIM   = 128
NUM_LAYERS   = 2
DROPOUT      = 0.3
N_SEEDS      = 3              # for seed ensembling (change H)
USE_POS_WEIGHT = False        # plain BCE for AUC (change C)
NEG_POS_RATIO = 10
N_UNDERSAMPLE_BAGS = 5
N_SEEDS = 1   # start here; bags already give ensemble diversity

# ----- Reproducibility -----
torch.manual_seed(SEED); np.random.seed(SEED)

# ----- Device selection -----
if torch.backends.mps.is_available():
    device = torch.device('mps')
elif torch.cuda.is_available():
    device = torch.device('cuda')
else:
    device = torch.device('cpu')

print(f'PyTorch {torch.__version__}  device={device}')


In [ ]:
import torch

print(torch.__version__)
print(torch.version.cuda)
print(torch.cuda.get_device_name(0))
print(torch.cuda.get_device_capability(0))
print(torch.cuda.get_arch_list())

x = torch.randn(8, device="cuda")
print(x.shape)

## 5. Build per-UID windows on the combined frame

In [ ]:
# import numpy as np

# DIR = "/kaggle/input/datasets/bachhoviet/split-data-npy"

# def load(name, mmap=True):
#     return np.load(f"{DIR}/{name}.npy", mmap_mode='r' if mmap else None)

# # Big arrays — memory-mapped (zero-copy, near-instant load)
# X_train_seq  = load("X_train_seq")
# X_test_seq   = load("X_test_seq")

# # Small arrays — load fully into RAM (mmap doesn't help, and you index them a lot)
# L_train      = load("L_train",      mmap=False)
# L_test       = load("L_test",       mmap=False)
# train_order  = load("train_order",  mmap=False)
# test_order   = load("test_order",   mmap=False)
# train_pos    = load("train_pos",    mmap=False)
# test_pos     = load("test_pos",     mmap=False)
# y_aligned    = load("y_aligned",    mmap=False)
# dt_m_aligned = load("dt_m_aligned", mmap=False)

# print(f"X_train_seq: {X_train_seq.shape} {X_train_seq.dtype}  (mmap)")
# print(f"X_test_seq : {X_test_seq.shape}  {X_test_seq.dtype}   (mmap)")
# print(f"y_aligned  : {y_aligned.shape}   {y_aligned.dtype}")

## 6. PyTorch `Dataset` and `DataLoader`

A thin wrapper around the numpy arrays. We hand the Dataset whatever subset of indices
the current fold needs, so we don't have to copy big arrays.


In [ ]:
# class WindowDataset(Dataset):
#     def __init__(self, X, lengths, y=None):
#         self.X = torch.from_numpy(X)
#         self.lengths = torch.from_numpy(lengths.astype('int64'))
#         self.y = None if y is None else torch.from_numpy(y.astype('float32'))

#     def __len__(self):
#         return self.X.shape[0]

#     def __getitem__(self, i):
#         if self.y is None:
#             return self.X[i], self.lengths[i]
#         return self.X[i], self.lengths[i], self.y[i]


# def make_loader(X, lengths, y, batch_size, shuffle):
#     return DataLoader(
#         WindowDataset(X, lengths, y),
#         batch_size=batch_size,
#         shuffle=shuffle,
#         num_workers=0,
#         pin_memory=False,
#         drop_last=False,
#     )

In [ ]:
"""
Layout B v2 — Conv1D + ResNet1D + STACKED Self-Attention + CLS + Mean/Max Pool + Static Tower.

Upgrades over fraud_cnn_resnet_attention.py:
  (1) Masked Mean + Max pool concatenated     → captures average + spike behavior
  (2) Learnable CLS token                     → transformer-style classification readout
  (3) Static-row tower (last transaction MLP) → recovers XGBoost-style intra-row signal
  (3) Stack of N transformer encoder blocks   → multi-hop attention reasoning

Same forward signature as v1:
    forward(x, lengths) -> logits of shape (B, 1)

Pipeline:
    (B, T, F)
        ├── transpose ───────────────────────► (B, F, T)
        ├── Conv1D stem (F → C, k=3) ────────► (B, C, T)
        ├── ResBlock1D × N_RESBLOCKS ────────► (B, C, T)
        ├── transpose + positional emb ──────► (B, T, C)
        ├── [optionally prepend CLS token] ──► (B, T+1, C)
        ├── TransformerBlock × N_ATTN_LAYERS ► (B, T+1, C)
        ├── pooling:
        │     - mean over real timesteps     → (B, C)
        │     - max  over real timesteps     → (B, C)        [optional]
        │     - CLS readout                  → (B, C)        [optional]
        │     - static MLP on last row       → (B, S)        [optional]
        ├── concatenate the parts            ► (B, pool_dim)
        └── MLP head                         ► (B, 1)
"""

import torch
from torch import nn


# =============================================================================
# 1D Residual Block — shape preserving (unchanged from v1)
# =============================================================================
class ResBlock1D(nn.Module):
    def __init__(self, c: int, k: int = 3, drop: float = 0.1):
        super().__init__()
        p = k // 2
        self.conv1 = nn.Conv1d(c, c, kernel_size=k, padding=p)
        self.bn1   = nn.BatchNorm1d(c)
        self.conv2 = nn.Conv1d(c, c, kernel_size=k, padding=p)
        self.bn2   = nn.BatchNorm1d(c)
        self.drop  = nn.Dropout(drop)
        self.act   = nn.ReLU()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        identity = x                                   # SKIP BRANCH
        out = self.act(self.bn1(self.conv1(x)))
        out = self.drop(out)
        out = self.bn2(self.conv2(out))
        out = out + identity                           # SKIP CONNECTION
        return self.act(out)


# =============================================================================
# Transformer encoder block — one self-attention + FF, both with residual+LN
# =============================================================================
class TransformerBlock(nn.Module):
    def __init__(self, c: int, n_heads: int, drop: float = 0.2):
        super().__init__()
        self.attn  = nn.MultiheadAttention(
            embed_dim=c, num_heads=n_heads, dropout=drop, batch_first=True,
        )
        self.norm1 = nn.LayerNorm(c)
        self.ff    = nn.Sequential(
            nn.Linear(c, c * 2),
            nn.GELU(),
            nn.Dropout(drop),
            nn.Linear(c * 2, c),
        )
        self.norm2 = nn.LayerNorm(c)

    def forward(self, h: torch.Tensor, key_padding_mask: torch.Tensor) -> torch.Tensor:
        a, _ = self.attn(h, h, h,
                         key_padding_mask=key_padding_mask,
                         need_weights=False)
        h = self.norm1(h + a)
        h = self.norm2(h + self.ff(h))
        return h


# =============================================================================
# Full v2 model
# =============================================================================
class FraudCNNResAttnV2(nn.Module):
    """
    Args
    ----
    n_features        : number of features per timestep (e.g. 244)
    window            : sequence length T (e.g. 20)
    c_hidden          : channel width of Conv stem / ResNet / attention
    n_resblocks       : number of ResBlock1D in the CNN stack
    n_attn_layers     : number of TransformerBlock layers (was 1 in v1)        ◄── NEW
    n_heads           : self-attention heads (must divide c_hidden)
    drop              : dropout used in ResBlocks, attention, FF, head, static tower
    static_hidden     : hidden width of the static-row tower
    use_cls           : prepend a learnable CLS token, use its embedding for readout ◄── NEW
    use_max_pool      : concat masked-max pool with mean pool                      ◄── NEW
    use_static_tower  : run the last raw row through an MLP and concat it          ◄── NEW
    output_dim        : final logit dim (keep =1 for BCEWithLogitsLoss)
    """
    def __init__(self,
                 n_features: int,
                 window: int = 20,
                 c_hidden: int = 128,
                 n_resblocks: int = 3,
                 n_attn_layers: int = 2,
                 n_heads: int = 4,
                 drop: float = 0.2,
                 static_hidden: int = 64,
                 use_cls: bool = True,
                 use_max_pool: bool = True,
                 use_static_tower: bool = True,
                 output_dim: int = 1):
        super().__init__()
        assert c_hidden % n_heads == 0, "c_hidden must be divisible by n_heads"

        self.window           = window
        self.use_cls          = use_cls
        self.use_max_pool     = use_max_pool
        self.use_static_tower = use_static_tower

        # ------- Stage 1 — Conv1D stem -------
        self.stem = nn.Sequential(
            nn.Conv1d(n_features, c_hidden, kernel_size=3, padding=1),
            nn.BatchNorm1d(c_hidden),
            nn.ReLU(),
        )

        # ------- Stage 2 — ResNet1D stack -------
        self.resblocks = nn.Sequential(*[
            ResBlock1D(c_hidden, k=3, drop=drop)
            for _ in range(n_resblocks)
        ])

        # ------- Stage 3 — positional embedding (over the WINDOW positions only;
        #                  the CLS token is added on top after this)
        self.pos = nn.Parameter(torch.zeros(1, window, c_hidden))
        nn.init.trunc_normal_(self.pos, std=0.02)

        # ------- (Optional) CLS token -------
        if use_cls:
            self.cls = nn.Parameter(torch.zeros(1, 1, c_hidden))
            nn.init.trunc_normal_(self.cls, std=0.02)

        # ------- Stage 4 — STACK of Transformer blocks -------
        self.attn_blocks = nn.ModuleList([
            TransformerBlock(c_hidden, n_heads, drop=drop)
            for _ in range(n_attn_layers)
        ])

        # ------- (Optional) Static-row tower -------
        if use_static_tower:
            self.static_mlp = nn.Sequential(
                nn.Linear(n_features, 256), nn.ReLU(), nn.Dropout(drop),
                nn.Linear(256, static_hidden), nn.ReLU(),
            )

        # ------- Compute pool dim from the toggles -------
        pool_dim = c_hidden                       # mean is always present
        if use_max_pool:    pool_dim += c_hidden
        if use_cls:         pool_dim += c_hidden
        if use_static_tower:pool_dim += static_hidden

        # ------- Stage 5 — Head -------
        self.head = nn.Sequential(
            nn.Dropout(drop),
            nn.Linear(pool_dim, 64), nn.ReLU(),
            nn.Dropout(drop),
            nn.Linear(64, output_dim),
        )

    # ----------------------------------------------------------------------
    def forward(self, x: torch.Tensor, lengths: torch.Tensor) -> torch.Tensor:
        # x: (B, T, F)  left-padded     lengths: (B,) real-step count
        B, T, F = x.shape

        # Keep last (most-recent) raw row aside for the static tower.
        # Left-padding writes to the RIGHT end, so x[:, -1, :] is always real.
        last_row = x[:, -1, :]                           # (B, F)

        # ----- CNN path (B, T, F) → (B, T, C) -----
        x_t = x.transpose(1, 2)                          # (B, F, T)
        h   = self.stem(x_t)                             # (B, C, T)
        h   = self.resblocks(h)                          # (B, C, T)
        h   = h.transpose(1, 2)                          # (B, T, C)
        h   = h + self.pos                               # positional encoding (over T only)

        # ----- Build real-position mask from `lengths` -----
        idx          = torch.arange(T, device=h.device).unsqueeze(0)   # (1, T)
        pos_from_end = T - 1 - idx                                     # (1, T)
        real         = pos_from_end < lengths.unsqueeze(1)             # (B, T) — True = real

        # ----- (Optional) Prepend CLS token -----
        if self.use_cls:
            cls = self.cls.expand(B, -1, -1)             # (B, 1, C)
            h   = torch.cat([cls, h], dim=1)             # (B, T+1, C)
            # CLS is always "real" so attention won't mask it
            cls_real = torch.ones(B, 1, dtype=torch.bool, device=h.device)
            real_full = torch.cat([cls_real, real], dim=1)              # (B, T+1)
            key_padding_mask = ~real_full
        else:
            key_padding_mask = ~real

        # ----- Stacked Transformer blocks -----
        for blk in self.attn_blocks:
            h = blk(h, key_padding_mask)

        # ----- Split CLS embedding from the time tokens -----
        if self.use_cls:
            cls_out = h[:, 0, :]                         # (B, C)
            seq_h   = h[:, 1:, :]                        # (B, T, C)
        else:
            seq_h = h                                    # (B, T, C)

        # ----- Pool over REAL timesteps only -----
        mask_f = real.unsqueeze(-1).float()              # (B, T, 1)
        cnt    = mask_f.sum(dim=1).clamp(min=1.0)        # (B, 1)

        pooled_parts = [(seq_h * mask_f).sum(dim=1) / cnt]   # mean

        if self.use_max_pool:
            # mask out padded positions with -inf so they never win the max
            seq_h_for_max = seq_h.masked_fill(~real.unsqueeze(-1), float('-inf'))
            pooled_parts.append(seq_h_for_max.max(dim=1).values)

        if self.use_cls:
            pooled_parts.append(cls_out)

        if self.use_static_tower:
            pooled_parts.append(self.static_mlp(last_row))

        pooled = torch.cat(pooled_parts, dim=1)          # (B, pool_dim)
        return self.head(pooled)                         # (B, 1)

In [ ]:
CNN_CHANNELS     = 128
N_RESBLOCKS      = 3
N_ATTN_LAYERS    = 1          # NEW — was effectively 1 in v1
N_HEADS          = 4
ATTN_DROPOUT     = 0.2
STATIC_HIDDEN    = 64         # NEW
USE_CLS          = True       # NEW
USE_MAX_POOL     = True       # NEW
USE_STATIC_TOWER = True       # NEW

In [ ]:
# # smoke test
# N_FEATURES = X_train_seq.shape[2]
# m  = FraudCNNResAttnV2(N_FEATURES).to(device)
# xb = torch.randn(8, WINDOW, N_FEATURES, device=device)
# lb = torch.randint(1, WINDOW + 1, (8,), device=device)
# print('forward smoke test out shape:', m(xb, lb).shape)   # expect torch.Size([8, 1])
# print('trainable params:', sum(p.numel() for p in m.parameters() if p.requires_grad))
# del m, xb, lb

## 8. Expanding-window CV + training loop

Same fold scheme as the Keras notebook. Training loop mirrors arunmohan003's pattern:

1. Per epoch, iterate batches and reset hidden state for each batch (windows are
   independent).
2. `optimizer.zero_grad()` → `forward` → `loss.backward()` → `clip_grad_norm_` →
   `optimizer.step()`.
3. Validation pass with `model.eval()` and `torch.no_grad()`.
4. Early-stop on validation AUC, restore best weights.


In [ ]:
def make_undersampled_idx(idxT, y, neg_pos_ratio=50, seed=42):
    rng = np.random.default_rng(seed)
    idxT = np.asarray(idxT)

    pos_idx = idxT[y[idxT] == 1]
    neg_idx = idxT[y[idxT] == 0]

    if len(pos_idx) == 0:
        raise ValueError("No fraud rows in this training fold.")

    n_neg_keep = min(len(neg_idx), int(neg_pos_ratio * len(pos_idx)))
    neg_keep = rng.choice(neg_idx, size=n_neg_keep, replace=False)

    sampled_idx = np.concatenate([pos_idx, neg_keep])
    rng.shuffle(sampled_idx)
    return sampled_idx


def ratio_msg(idx, y):
    pos = int((y[idx] == 1).sum())
    neg = int(len(idx) - pos)
    return f"pos={pos:,}, neg={neg:,}, neg/pos={neg / max(pos, 1):.1f}:1"

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available()
                       else 'mps' if torch.backends.mps.is_available()
                       else 'cpu')
print('device:', device)


def expanding_month_folds(months_array, min_train_months=MIN_TRAIN_MONTHS):
    months = sorted(np.unique(months_array).tolist())
    for vm in months[min_train_months:]:
        tm = [m for m in months if m < vm]
        ti = np.flatnonzero(np.isin(months_array, tm))
        vi = np.flatnonzero(months_array == vm)
        yield (vm, tm, ti, vi)

In [ ]:
# def train_one_fold(X_tr, L_tr, y_tr, X_va, L_va, y_va, n_features,
#                    epochs, batch, lr, weight_decay, device,
#                    early_stop_patience, grad_clip, seed):
#     torch.manual_seed(seed); np.random.seed(seed)
#     # model = FraudCNNResAttn(n_features).to(device)
#     model = FraudCNNResAttnV2(
#         n_features,
#         c_hidden=CNN_CHANNELS,
#         n_resblocks=N_RESBLOCKS,
#         n_attn_layers=N_ATTN_LAYERS,
#         n_heads=N_HEADS,
#         drop=ATTN_DROPOUT,
#         static_hidden=STATIC_HIDDEN,
#         use_cls=USE_CLS,
#         use_max_pool=USE_MAX_POOL,
#         use_static_tower=USE_STATIC_TOWER,
#     ).to(device) # CNN_ResNet_Attention v2

#     if USE_POS_WEIGHT:
#         pos = float((y_tr == 1).sum()); neg = float(len(y_tr) - pos)
#         pw = torch.tensor([np.sqrt(neg / max(pos, 1.0))], device=device, dtype=torch.float32)
#         loss_fn = nn.BCEWithLogitsLoss(pos_weight=pw)
#     else:
#         loss_fn = nn.BCEWithLogitsLoss()                            # plain BCE → better AUC

#     optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)

#     train_loader = make_loader(X_tr, L_tr, y_tr, batch_size=batch, shuffle=True)
#     val_loader   = make_loader(X_va, L_va, y_va, batch_size=batch, shuffle=False)

#     steps = max(1, math.ceil(len(X_tr) / batch))
#     scheduler = torch.optim.lr_scheduler.OneCycleLR(
#         optimizer, max_lr=lr, epochs=epochs, steps_per_epoch=steps,
#         pct_start=0.1, anneal_strategy='cos',
#     )

#     best_auc, best_state, best_val_preds, bad = -1.0, None, None, 0
#     for epoch in range(1, epochs + 1):
#         model.train()
#         t0 = time.time(); running, n_seen = 0.0, 0
#         for xb, lb, yb in train_loader:
#             xb = xb.to(device=device, dtype=torch.float32)
#             lb = lb.to(device=device, dtype=torch.long)
#             yb = yb.to(device=device, dtype=torch.float32)
#             optimizer.zero_grad()
#             logits = model(xb, lb).squeeze(-1)
#             loss = loss_fn(logits, yb)
#             loss.backward()
#             nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
#             optimizer.step(); scheduler.step()
#             running += loss.item() * xb.size(0); n_seen += xb.size(0)
#         train_loss = running / max(n_seen, 1)

#         model.eval(); preds = []
#         with torch.no_grad():
#             for xb, lb, _ in val_loader:
#                 xb = xb.to(device=device, dtype=torch.float32)
#                 lb = lb.to(device=device, dtype=torch.long)
#                 logits = model(xb, lb).squeeze(-1)
#                 preds.append(torch.sigmoid(logits).cpu().numpy())
#         val_preds = np.concatenate(preds)
#         val_auc = roc_auc_score(y_va, val_preds)

#         print(f'   ep {epoch:>2}/{epochs}  loss={train_loss:.4f}  val_auc={val_auc:.4f}  ({time.time()-t0:.1f}s)')
#         if val_auc > best_auc:
#             best_auc = val_auc; best_state = copy.deepcopy(model.state_dict())
#             best_val_preds = val_preds; bad = 0
#         else:
#             bad += 1
#             if bad >= early_stop_patience:
#                 print(f'   early stop at epoch {epoch}'); break

#     if best_state is not None: model.load_state_dict(best_state)
#     return best_val_preds, best_auc, model

In [ ]:
# # ===== run folds with seed ensembling =====
# oof        = np.full(len(X_train_seq), np.nan, dtype=np.float32)
# test_preds = np.zeros(len(X_test_seq), dtype=np.float32)
# fold_aucs  = []

# fold_specs = list(expanding_month_folds(dt_m_aligned, MIN_TRAIN_MONTHS))
# print(f'{len(fold_specs)} folds; {N_SEEDS} seeds per fold')

# for fold, (vm, tm, idxT, idxV) in enumerate(fold_specs):
#     print(f'\n=== Fold {fold}: train {tm} → validate {vm} '
#           f'(train={len(idxT):,}, valid={len(idxV):,}) ===')
#     print("full train:", ratio_msg(idxT, y_aligned))

#     bag_val_preds, bag_test_preds = [], []

#     for bag in range(N_UNDERSAMPLE_BAGS):
#         seed = SEED + 1000 * fold + bag
#         idxT_sub = make_undersampled_idx(
#             idxT,
#             y_aligned,
#             neg_pos_ratio=NEG_POS_RATIO,
#             seed=seed,
#         )

#         print(f'-- bag {bag+1}/{N_UNDERSAMPLE_BAGS}, seed={seed} --')
#         print("sampled train:", ratio_msg(idxT_sub, y_aligned))

#         vp, va, model = train_one_fold(
#             X_train_seq[idxT_sub], L_train[idxT_sub], y_aligned[idxT_sub],
#             X_train_seq[idxV],    L_train[idxV],    y_aligned[idxV],
#             n_features=N_FEATURES, epochs=EPOCHS, batch=BATCH,
#             lr=LR, weight_decay=WEIGHT_DECAY, device=device,
#             early_stop_patience=EARLY_STOP_PATIENCE,
#             grad_clip=GRAD_CLIP, seed=seed,
#         )
#         bag_val_preds.append(vp)

#         model.eval()
#         test_loader = make_loader(X_test_seq, L_test, y=None, batch_size=BATCH, shuffle=False)
#         tps = []
#         with torch.no_grad():
#             for xb, lb in test_loader:
#                 xb = xb.to(device=device, dtype=torch.float32)
#                 lb = lb.to(device=device, dtype=torch.long)
#                 logits = model(xb, lb).squeeze(-1)
#                 tps.append(torch.sigmoid(logits).cpu().numpy())

#         bag_test_preds.append(np.concatenate(tps))

#         del model
#         if device.type == 'mps':
#             torch.mps.empty_cache()
#         gc.collect()

#     fold_val  = np.mean(bag_val_preds, axis=0)
#     fold_test = np.mean(bag_test_preds, axis=0)
#     fold_auc   = roc_auc_score(y_aligned[idxV], fold_val)
#     print(f'   fold AUC (seed-avg) = {fold_auc:.4f}')
#     fold_aucs.append((int(vm), float(fold_auc)))
#     oof[idxV]  = fold_val
#     test_preds += fold_test

# if len(fold_specs):
#     test_preds /= len(fold_specs)

# validated = ~np.isnan(oof)
# overall_auc = roc_auc_score(y_aligned[validated], oof[validated])
# print(f'\n=== LSTM OOF AUC (validated months only) = {overall_auc:.4f} ===')
# print(f'   per-fold: {fold_aucs}')

## SPLITTING BY UIDS-DISJOINT

In [ ]:
from pathlib import Path
import numpy as np

CACHE_DIR = Path("/kaggle/input/datasets/bachhoviet/v4-dataset")
def load(name, mmap=True):
    return np.load(CACHE_DIR / f"{name}.npy", mmap_mode="r" if mmap else None)

def load_orig_list(prefix):
    flat = load(f"{prefix}_flat", mmap=False)
    offsets = load(f"{prefix}_offsets", mmap=False)
    return [flat[offsets[i]:offsets[i + 1]] for i in range(len(offsets) - 1)]

# Big arrays
X_tr = load("X_tr").astype("float32")
X_va = load("X_va").astype("float32")

# Small arrays
L_tr = load("L_tr", mmap=False)
Y_tr = load("Y_tr", mmap=False)
M_tr = load("M_tr", mmap=False)
uids_tr = load("uids_tr", mmap=False)

L_va = load("L_va", mmap=False)
Y_va = load("Y_va", mmap=False)
M_va = load("M_va", mmap=False)
uids_va = load("uids_va", mmap=False)

# v4 orig lists
orig_tr = load_orig_list("orig_tr")
orig_va = load_orig_list("orig_va")

## IF LEFT PADDING

In [ ]:
def right_to_left_padded(X, L, Y=None, M=None):
    """
    Convert right-padded arrays to left-padded arrays while preserving timestep order.

    X: (N, T, F)
    L: (N,)
    Y: optional (N, T)
    M: optional (N, T)

    Returns:
      X_left, Y_left, M_left
    """
    N, T = X.shape[:2]

    X_left = np.zeros_like(X)
    Y_left = None if Y is None else np.zeros_like(Y)
    M_left = None if M is None else np.zeros_like(M)

    for i, length in enumerate(L.astype(int)):
        if length <= 0:
            continue

        X_left[i, T - length:T, :] = X[i, :length, :]

        if Y is not None:
            Y_left[i, T - length:T] = Y[i, :length]

        if M is not None:
            M_left[i, T - length:T] = M[i, :length]

    return X_left, Y_left, M_left

In [ ]:
X_tr_left, Y_tr_left, M_tr_left = right_to_left_padded(X_tr, L_tr, Y_tr, M_tr)
X_va_left, Y_va_left, M_va_left = right_to_left_padded(X_va, L_va, Y_va, M_va)

X_tr, Y_tr, M_tr = X_tr_left, Y_tr_left, M_tr_left
X_va, Y_va, M_va = X_va_left, Y_va_left, M_va_left

In [ ]:
class FraudCNNResAttnV2PerStep(nn.Module):
    """
    Per-timestep CNN/ResNet/Attention model.

    Input : X shape (B, T, F), left-padded
            lengths shape (B,)
    Output: logits shape (B, T)

    Train with masked BCE:
        loss = BCE(logits[M], Y[M])
    """
    def __init__(self,
                 n_features: int,
                 window: int = 20,
                 c_hidden: int = 128,
                 n_resblocks: int = 3,
                 n_attn_layers: int = 2,
                 n_heads: int = 4,
                 drop: float = 0.2):
        super().__init__()

        assert c_hidden % n_heads == 0

        self.window = window

        self.stem = nn.Sequential(
            nn.Conv1d(n_features, c_hidden, kernel_size=3, padding=1),
            nn.BatchNorm1d(c_hidden),
            nn.ReLU(),
        )

        self.resblocks = nn.Sequential(*[
            ResBlock1D(c_hidden, k=3, drop=drop)
            for _ in range(n_resblocks)
        ])

        self.pos = nn.Parameter(torch.zeros(1, window, c_hidden))
        nn.init.trunc_normal_(self.pos, std=0.02)

        self.attn_blocks = nn.ModuleList([
            TransformerBlock(c_hidden, n_heads, drop=drop)
            for _ in range(n_attn_layers)
        ])

        # One logit per timestep
        self.token_head = nn.Sequential(
            nn.Dropout(drop),
            nn.Linear(c_hidden, 64),
            nn.ReLU(),
            nn.Dropout(drop),
            nn.Linear(64, 1),
        )

    def forward(self, x, lengths):
        # x: (B, T, F), LEFT-padded
        B, T, _ = x.shape

        h = x.transpose(1, 2)        # (B, F, T)
        h = self.stem(h)             # (B, C, T)
        h = self.resblocks(h)        # (B, C, T)
        h = h.transpose(1, 2)        # (B, T, C)

        h = h + self.pos[:, :T, :]

        # left-padding mask: real positions are at the RIGHT end
        idx = torch.arange(T, device=x.device).unsqueeze(0)
        real = (T - 1 - idx) < lengths.to(x.device).unsqueeze(1)
        key_padding_mask = ~real

        for blk in self.attn_blocks:
            h = blk(h, key_padding_mask)

        logits = self.token_head(h).squeeze(-1)   # (B, T)
        return logits

In [ ]:
class UIDStepDataset(Dataset):
    def __init__(self, X, L, Y, M):
        self.X = torch.from_numpy(X)
        self.L = torch.from_numpy(L.astype("int64"))
        self.Y = torch.from_numpy(Y.astype("float32"))
        self.M = torch.from_numpy(M.astype(bool))

    def __len__(self):
        return self.X.shape[0]

    def __getitem__(self, i):
        return self.X[i], self.L[i], self.Y[i], self.M[i]


def make_uid_step_loader(X, L, Y, M, batch_size, shuffle):
    return DataLoader(
        UIDStepDataset(X, L, Y, M),
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=0,
        pin_memory=False,
        drop_last=False,
    )

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available()
                       else 'mps' if torch.backends.mps.is_available()
                       else 'cpu')
print('device:', device)

# Hyperparameters for the per-UID setup
BATCH               = 128        # SMALL: each sample is (MAX_LEN, F)
EPOCHS              = 30
LR                  = 1e-3
WEIGHT_DECAY        = 2e-4
GRAD_CLIP           = 1.0
EARLY_STOP_PATIENCE = 6
HIDDEN_DIM          = 128
NUM_LAYERS          = 2
DROPOUT             = 0.3
BIDIRECTIONAL       = False
N_SEEDS             = 10

In [ ]:
def masked_bce(logits, targets, mask):
    return nn.functional.binary_cross_entropy_with_logits(
        logits[mask],
        targets[mask],
    )

In [ ]:
MAX_LEN = X_tr.shape[1]
WINDOW = MAX_LEN
N_FEATURES = X_tr.shape[2]

assert X_tr.shape[:2] == Y_tr.shape == M_tr.shape
assert X_va.shape[:2] == Y_va.shape == M_va.shape
assert L_tr.max() <= MAX_LEN
assert L_va.max() <= MAX_LEN

print("MAX_LEN:", MAX_LEN)
print("N_FEATURES:", N_FEATURES)
print("X_tr:", X_tr.shape, X_tr.dtype)
print("Y_tr:", Y_tr.shape, Y_tr.dtype)
print("M_tr:", M_tr.shape, M_tr.dtype)

In [ ]:
def train_one_uid_run(X_tr, L_tr, Y_tr, M_tr,
                      X_va, L_va, Y_va, M_va,
                      n_features, epochs, batch, lr, weight_decay,
                      device, patience, grad_clip, seed):
    torch.manual_seed(seed); np.random.seed(seed)
    model = FraudCNNResAttnV2PerStep(
        n_features=n_features,
        window=MAX_LEN,
        c_hidden=CNN_CHANNELS,
        n_resblocks=N_RESBLOCKS,
        n_attn_layers=N_ATTN_LAYERS,
        n_heads=N_HEADS,
        drop=ATTN_DROPOUT,
    ).to(device)

    train_loader = make_uid_step_loader(X_tr, L_tr, Y_tr, M_tr, batch_size=batch, shuffle=True)
    val_loader   = make_uid_step_loader(X_va, L_va, Y_va, M_va, batch_size=batch, shuffle=False)

    opt   = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    sched = torch.optim.lr_scheduler.OneCycleLR(
        opt, max_lr=lr, epochs=epochs,
        steps_per_epoch=max(1, math.ceil(len(X_tr) / batch)),
        pct_start=0.1, anneal_strategy='cos',
    )

    best_auc, best_state, best_val_flat, bad = -1.0, None, None, 0
    for epoch in range(1, epochs + 1):
        model.train(); running, n_seen = 0.0, 0; t0 = time.time()
        for xb, lb, yb, mb in train_loader:
            xb = xb.to(device=device, dtype=torch.float32)
            lb = lb.to(device)
            yb = yb.to(device=device, dtype=torch.float32)
            mb = mb.to(device)
            opt.zero_grad()
            logits = model(xb, lb)
            loss   = masked_bce(logits, yb, mb)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
            opt.step(); sched.step()
            running += loss.item() * xb.size(0); n_seen += xb.size(0)
        train_loss = running / max(n_seen, 1)

        model.eval(); all_p, all_y = [], []
        with torch.no_grad():
            for xb, lb, yb, mb in val_loader:
                xb = xb.to(device=device, dtype=torch.float32)
                lb = lb.to(device)
                p = torch.sigmoid(model(xb, lb)).cpu().numpy()
                mn = mb.numpy()
                all_p.append(p[mn])
                all_y.append(yb.numpy()[mn])
        val_p = np.concatenate(all_p); val_y = np.concatenate(all_y)
        val_auc = roc_auc_score(val_y, val_p)

        print(f'   ep {epoch:>2}/{epochs}  loss={train_loss:.4f}  '
              f'val_auc={val_auc:.4f}  ({time.time()-t0:.1f}s)')

        if val_auc > best_auc:
            best_auc      = val_auc
            best_state    = copy.deepcopy(model.state_dict())
            best_val_flat = val_p
            bad = 0
        else:
            bad += 1
            if bad >= patience:
                print(f'   early stop at epoch {epoch}'); break

    if best_state is not None: model.load_state_dict(best_state)
    return best_val_flat, best_auc, model

In [ ]:
print(f'Per-UID run with {N_SEEDS} seeds')
print(f'  train UIDs={X_tr.shape[0]:,}  val UIDs={X_va.shape[0]:,}')

seed_val_preds, seed_aucs = [], []
SEED = 42
for s in range(N_SEEDS):
    seed = SEED + s
    print(f'\n-- seed {seed} --')
    val_p, val_auc, model = train_one_uid_run(
        X_tr, L_tr, Y_tr, M_tr,
        X_va, L_va, Y_va, M_va,
        n_features=N_FEATURES,
        epochs=EPOCHS, batch=BATCH, lr=LR,
        weight_decay=WEIGHT_DECAY, device=device,
        patience=EARLY_STOP_PATIENCE, grad_clip=GRAD_CLIP, seed=seed,
    )
    seed_val_preds.append(val_p); seed_aucs.append(val_auc)
    del model
    if device.type == 'mps': torch.mps.empty_cache()
    gc.collect()

avg_val_pred = np.mean(seed_val_preds, axis=0)

# Re-construct val labels in the same flattened order as the predictions
val_y_flat = []
for xb_idx in range(X_va.shape[0]):
    val_y_flat.append(Y_va[xb_idx][M_va[xb_idx]])
val_y_flat = np.concatenate(val_y_flat)

overall_auc = roc_auc_score(val_y_flat, avg_val_pred)
print(f'\n=== Per-seed AUCs: {[round(a,4) for a in seed_aucs]}')
print(f'=== Seed-averaged OOF AUC = {overall_auc:.4f} ===')

In [ ]:
# Map per-step predictions back to per-transaction TransactionIDs
flat_orig_ids = np.concatenate([
    orig_va[i][:int(L_va[i])]                  # original TransactionIDs, in time order
    for i in range(len(orig_va))
])

assert len(flat_orig_ids) == len(avg_val_pred), \
    f'len mismatch: {len(flat_orig_ids)} vs {len(avg_val_pred)}'

oof_df = pd.DataFrame({
    'TransactionID':   flat_orig_ids,
    'oof_lstm_per_uid': avg_val_pred,
    'isFraud_label':    val_y_flat,
})
# oof_df.to_csv('oof_lstm_per_uid.csv', index=False)
# print(f'Saved {len(oof_df):,} OOF rows to oof_lstm_per_uid.csv')
print(f'  overall AUC: {roc_auc_score(oof_df.isFraud_label, oof_df.oof_lstm_per_uid):.4f}')